<a href="https://colab.research.google.com/github/AstonVSabrido/Final-Project-CS2/blob/main/Kamia_Transport_NB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import requests
import json
url = "https://raw.githubusercontent.com/AstonVSabrido/Final-Project-CS2/refs/heads/main/transport.json"
response = requests.get(url)
trips = response.json()
print(trips)


[{'trip_id': 101, 'route_id': 1, 'origin': 'Davao City', 'destination': 'Tagum', 'date': '2025-09-20', 'departure_time': '08:00', 'arrival_time': '09:30', 'travel_duration_min': 90, 'bus_operator': 'Davao Express', 'bus_type': 'Aircon', 'capacity': 45, 'passengers': 35, 'occupancy_rate': 77.8, 'fare': {'regular': 120, 'student': 96, 'senior': 84}, 'status': 'On Time'}, {'trip_id': 102, 'route_id': 1, 'origin': 'Davao City', 'destination': 'Tagum', 'date': '2025-09-20', 'departure_time': '14:00', 'arrival_time': '15:40', 'travel_duration_min': 100, 'bus_operator': 'Mindanao Transport', 'bus_type': 'Regular', 'capacity': 50, 'passengers': 48, 'occupancy_rate': 96.0, 'fare': {'regular': 110, 'student': 88, 'senior': 77}, 'status': 'Delayed'}, {'trip_id': 201, 'route_id': 2, 'origin': 'Davao City', 'destination': 'Mati', 'date': '2025-09-21', 'departure_time': '07:30', 'arrival_time': '11:00', 'travel_duration_min': 210, 'bus_operator': 'Davao Express', 'bus_type': 'Deluxe', 'capacity': 40

In [2]:
!pip install firebase-admin

In [3]:
import firebase_admin
from firebase_admin import credentials, db
# Load the private key
cred = credentials.Certificate("/content/firebase_key_project.json")
# Initialize the app with your database URL
firebase_admin.initialize_app(cred, {
 "databaseURL": "https://kamia-transport-db-default-rtdb.asia-southeast1.firebasedatabase.app/"
})
print("Firebase connected successfully!")


Firebase connected successfully!


In [10]:
import requests
import json

with open('/content/transport.json', 'r') as f:
    data = json.load(f)

print("JSON file loaded")
ref = db.reference("trips")
for trip in data:
  ref.child(str(trip["trip_id"])).set(trip)
print("Data uploaded successfully!")

JSON file loaded
Data uploaded successfully!


In [12]:
ref = db.reference('trips')
trips_data = ref.get()

if trips_data:
    # Convert the dictionary of trips into a list of trip dictionaries
    trips_list = list(trips_data.values())
    print("Trips data retrieved successfully:")
    print(f"Number of trips: {len(trips_list)}")
    print("First trip:")
    print(trips_list[0])
else:
    trips_list = []
    print("No trips data found in Firebase.")

Trips data retrieved successfully:
Number of trips: 6
First trip:
{'arrival_time': '09:30', 'bus_operator': 'Davao Express', 'bus_type': 'Aircon', 'capacity': 45, 'date': '2025-09-20', 'departure_time': '08:00', 'destination': 'Tagum', 'fare': {'regular': 120, 'senior': 84, 'student': 96}, 'occupancy_rate': 77.8, 'origin': 'Davao City', 'passengers': 35, 'route_id': 1, 'status': 'On Time', 'travel_duration_min': 90, 'trip_id': 101}


In [22]:
search_history = []

while True:
    print("\n--- Trip Filtering Application ---")
    print("1. Search for trips")
    print("2. View search history")
    print("3. Exit")

    choice = input("Enter your choice: ")

    if choice == '1':
        print("Searching...")
        valid_criteria = ['destination', 'bus_type', 'trip_id', 'status']
        while True:
            search_criterion_input = input(f"Enter search criterion ({', '.join(valid_criteria)}): ").lower()
            if search_criterion_input in valid_criteria:
                search_criterion = search_criterion_input
                print(f"Selected search criterion: {search_criterion}")
                break
            else:
                print("Invalid criterion. Please choose from the available options.")

        # Extract unique values for the selected criterion
        unique_values = sorted(list(set([trip[search_criterion] for trip in trips_list if search_criterion in trip]))) # Use trips_list here
        print(f"\nAvailable {search_criterion} values: {', '.join(map(str, unique_values))}")

        # Prompt user for search value
        search_value_input = input(f"Enter the {search_criterion} you are looking for: ")

        # Convert search_value if criterion is trip_id
        if search_criterion == 'trip_id':
            try:
                search_value = int(search_value_input)
            except ValueError:
                print("Invalid input for Trip ID. Please enter a number.")
                continue
        else:
            search_value = search_value_input

        # Filter trips
        filtered_trips = []
        for trip in trips_list:
            if search_criterion in trip and trip[search_criterion] == search_value:
                filtered_trips.append(trip)

        # Display filtered trips
        if filtered_trips:
            print(f"\nFound {len(filtered_trips)} trip(s) matching '{search_value}' for {search_criterion}:")
            for trip in filtered_trips:
                search_history.append(trip) # Add trip to search history
                print(json.dumps(trip, indent=2))
        else:
            print(f"\nNo trips found matching '{search_value}' for {search_criterion}.")


    elif choice == '2':
        print("Viewing history...")
        if not search_history:
            print("No searches have been performed yet.")
        else:
            print("\n--- Search History ---")
            for i, trip in enumerate(search_history):
                print(f"\n--- Search History Entry {i + 1} ---")
                print(json.dumps(trip, indent=2))
    elif choice == '3':
        print("Exiting the application. Goodbye!")
        break
    else:
        print("Invalid choice. Please enter 1, 2, or 3.")


--- Trip Filtering Application ---
1. Search for trips
2. View search history
3. Exit
Enter your choice: 1
Searching...
Enter search criterion (destination, bus_type, trip_id, status): destination
Selected search criterion: destination

Available destination values: Compostela, Mati, Tagum
Enter the destination you are looking for: Compostela

Found 2 trip(s) matching 'Compostela' for destination:
{
  "arrival_time": "10:15",
  "bus_operator": "Tagum Lines",
  "bus_type": "Regular",
  "capacity": 40,
  "date": "2025-09-22",
  "departure_time": "09:00",
  "destination": "Compostela",
  "fare": {
    "regular": 90,
    "senior": 63,
    "student": 72
  },
  "occupancy_rate": 50.0,
  "origin": "Tagum",
  "passengers": 20,
  "route_id": 3,
  "status": "Cancelled",
  "travel_duration_min": 75,
  "trip_id": 301
}
{
  "arrival_time": "18:20",
  "bus_operator": "Tagum Lines",
  "bus_type": "Aircon",
  "capacity": 38,
  "date": "2025-09-22",
  "departure_time": "17:00",
  "destination": "Compo